# XOPT - OR-LIBRARY p-MEDIAN WITH MIP

It solves every OR-Library p-median instance with `sklearn_extra.cluster.KMedoids` and reports runtime and gap to the published best-known value, and uses the `KMedoids` solution as a feasible warm start for the exact `python-mip` formulation and optionally adds the heuristic objective as a cutoff constraint.

### SETUP

In [1]:
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

Importing the libraries:

In [2]:
import gc
import os
import re
import sys
import math
import heapq

from pathlib         import Path
from time            import perf_counter
from IPython.display import display

import numpy  as np
import pandas as pd

from sklearn_extra.cluster import KMedoids
from mip                   import xsum, minimize, Model, BINARY, OptimizationStatus

Find the project root directory:

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / 'instances').exists() and \
           (candidate / 'notebooks').exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing 'instances' and 'notebooks'."
    )


def instance_sort_key(pathlike: str | Path) -> tuple[int, str]:
    stem  = Path(pathlike).stem
    match = re.search(r'(\d+)$', stem)

    if match is None:
        return (10**9, stem)

    return (int(match.group(1)), stem)


def parse_optional_int_env(name: str) -> int | None:
    raw_value = os.getenv(name)

    if raw_value is None:
        return None

    raw_value = raw_value.strip()
    if not raw_value:
        return None

    return int(raw_value)


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


### EXPERIMENT CONFIGURATION

In [4]:
INSTANCES_DIR = PROJECT_ROOT  / 'instances'
PMEDOPT_PATH  = INSTANCES_DIR / 'pmedopt.txt'
OUTPUT_DIR    = PROJECT_ROOT  / 'notebooks' / 'experiments_sbpo' / 'artifacts'

KMEDOIDS_OUTPUT_PATH   = OUTPUT_DIR / 'kmedoids_results.csv'
WARM_START_OUTPUT_PATH = OUTPUT_DIR / 'mip_warm_start_results.csv'

TIME_LIMIT_SECONDS     = parse_optional_int_env('PMED_TIME_LIMIT_SECONDS') or 180
MAX_INSTANCES          = parse_optional_int_env('PMED_MAX_INSTANCES'     )

INSTANCE_FILTER        = os.getenv('PMED_INSTANCE_FILTER')

KMEDOIDS_RANDOM_STATE  = parse_optional_int_env('PMED_KMEDOIDS_RANDOM_STATE') or 0
KMEDOIDS_MAX_ITER      = parse_optional_int_env('PMED_KMEDOIDS_MAX_ITER'    ) or 10
KMEDOIDS_METHOD        = os.getenv('PMED_KMEDOIDS_METHOD', 'pam'        )
KMEDOIDS_INIT          = os.getenv('PMED_KMEDOIDS_INIT'  , 'k-medoids++')

APPLY_HEURISTIC_CUTOFF = os.getenv('PMED_APPLY_HEURISTIC_CUTOFF', '1') != '0'


def list_orlibrary_instances(instances_dir: Path) -> list[str]:
    return sorted(
        [
            path.name
            for path in instances_dir.glob('pmed*.txt')
            if  path.name != 'pmedopt.txt'
        ],
        key=instance_sort_key,
    )


def apply_instance_selection(
    instance_names   : list[str],
    pattern          : str | None = None,
    limit            : int | None = None,
) -> list[str]:
    selected = list(instance_names)

    if pattern:
        regex    = re.compile(pattern)
        selected = [name for name in selected if regex.search(name)]

    if limit is not None:
        selected = selected[:limit]

    return selected


def load_best_known_costs(pmedopt_path: Path) -> dict[str, int]:
    best_known_costs: dict[str, int] = {}

    for raw_line in pmedopt_path.read_text().splitlines()[1:]:
        line = raw_line.strip()
        if not line:
            continue

        instance_id, value = line.split()[:2]

        best_known_costs[f'{instance_id}.txt'] = int(value)

    return best_known_costs


CANONICAL_INSTANCE_NAMES = [
    f'pmed{i}.txt' for i in range(1, 41)
]

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)
BEST_KNOWN_COSTS   = load_best_known_costs(PMEDOPT_PATH)

if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f'No p-median instances were selected from {INSTANCES_DIR}.'
    )

print(f'Instances directory  : {INSTANCES_DIR}')
print(f'Instances discovered : {len(ALL_INSTANCE_NAMES)}')
print(f'Instances selected   : {len(INSTANCE_NAMES    )}')

print()

print(f'Time limit per instance (s)     : {TIME_LIMIT_SECONDS    }')
print(f'KMedoids random_state           : {KMEDOIDS_RANDOM_STATE }')
print(f'KMedoids max_iter               : {KMEDOIDS_MAX_ITER     }')
print(f'Heuristic cutoff applied to MIP : {APPLY_HEURISTIC_CUTOFF}')
print(f'KMedoids method / init          : {KMEDOIDS_METHOD} / {KMEDOIDS_INIT}')

Instances directory  : /home/rei-luisinho/xopt/instances
Instances discovered : 40
Instances selected   : 40

Time limit per instance (s)     : 180
KMedoids random_state           : 0
KMedoids max_iter               : 10
Heuristic cutoff applied to MIP : True
KMedoids method / init          : pam / k-medoids++


In [5]:
missing_canonical = sorted(set(CANONICAL_INSTANCE_NAMES) - set(ALL_INSTANCE_NAMES      ), key=instance_sort_key)
extra_instances   = sorted(set(ALL_INSTANCE_NAMES      ) - set(CANONICAL_INSTANCE_NAMES), key=instance_sort_key)
missing_optima    = sorted(set(ALL_INSTANCE_NAMES      ) - set(BEST_KNOWN_COSTS        ), key=instance_sort_key)

if missing_canonical:
    print(f'Missing canonical OR-Library instances: {missing_canonical}')
else:
    print('All 40 canonical OR-Library instances are present.')

if extra_instances:
    print(f'Additional instances discovered    : {extra_instances}')

if missing_optima:
    print(f'Instances without best-known value : {missing_optima}')

print()
print('Instances to solve:')
print(', '.join(name.removesuffix('.txt') for name in INSTANCE_NAMES))

All 40 canonical OR-Library instances are present.

Instances to solve:
pmed1, pmed2, pmed3, pmed4, pmed5, pmed6, pmed7, pmed8, pmed9, pmed10, pmed11, pmed12, pmed13, pmed14, pmed15, pmed16, pmed17, pmed18, pmed19, pmed20, pmed21, pmed22, pmed23, pmed24, pmed25, pmed26, pmed27, pmed28, pmed29, pmed30, pmed31, pmed32, pmed33, pmed34, pmed35, pmed36, pmed37, pmed38, pmed39, pmed40


In [6]:
def read_orlibrary_graph(instance_path: Path) -> dict[str, object]:
    with instance_path.open() as file:
        header = file.readline().split()

        if len(header) < 3:
            raise ValueError(f'Could not parse instance header: {instance_path}')

        n, m, p = map(int, header[:3])

        adjacency_map: list[dict[int, int]] = [dict() for _ in range(n)]
        raw_edge_count         = 0
        duplicate_edge_updates = 0

        for raw_line in file:
            line = raw_line.strip()

            if not line:
                continue

            u, v, cost = map(int, line.split())
            u -= 1
            v -= 1

            if v in adjacency_map[u]:
                duplicate_edge_updates += 1

            adjacency_map[u][v] = cost
            adjacency_map[v][u] = cost
            raw_edge_count += 1

    if raw_edge_count != m:
        raise ValueError(
            f'Expected {m} edges in {instance_path.name}, but found {raw_edge_count}.'
        )

    adjacency = [
        list(neighbors.items())
        for neighbors in adjacency_map
    ]

    return {
        'n': n,
        'm': m,
        'p': p,

        'adjacency'              : adjacency             ,
        'duplicate_edge_updates' : duplicate_edge_updates,
    }


def all_pairs_shortest_paths(adjacency: list[list[tuple[int, int]]]) -> np.ndarray:
    n         = len(adjacency)
    distances = np.full((n, n), np.inf, dtype=np.float64)

    for source in range(n):
        row         = distances[source]
        row[source] = 0.0

        heap: list[tuple[float, int]] = [(0.0, source)]

        while heap:
            dist_u, u = heapq.heappop(heap)
            if dist_u > row[u]:
                continue

            for v, weight in adjacency[u]:
                candidate = dist_u + weight

                if candidate < row[v]:
                    row[v] = candidate

                    heapq.heappush(heap, (candidate, v))

        if np.isinf(row).any():
            raise ValueError(
                'Instance graph is disconnected. Could not compute all-pairs distances.'
            )

    return distances.astype(np.int64)

### MIP + WARM START

In [7]:
def finite_or_none(value: float | int | None) -> float | None:
    if value is None:
        return None

    value = float(value)

    if not math.isfinite(value):
        return None

    return value


def normalize_number(value: float | int | None, digits: int = 4):
    value = finite_or_none(value)

    if value is None:
        return None

    if abs(value - round(value)) < 10 ** (-digits):
        return int(round(value))

    return round(value, digits)


def gap_to_reference_percent(
    value     : float | int | None,
    reference : float | int | None,
) -> float | None:
    value     = finite_or_none(value    )
    reference = finite_or_none(reference)

    if value     is None or \
       reference is None or \
       reference == 0:
        return None

    return 100.0 * (value - reference) / reference


def gap_to_bound_percent(
    objective_value : float | int | None,
    objective_bound : float | int | None,
) -> float | None:
    objective_value = finite_or_none(objective_value)
    objective_bound = finite_or_none(objective_bound)

    if objective_value is None or \
       objective_bound is None or \
       objective_value == 0:
        return None

    return 100.0 * (objective_value - objective_bound) / objective_value


def speedup_factor(
    baseline_seconds  : float | int | None,
    candidate_seconds : float | int | None,
) -> float | None:
    baseline_seconds  = finite_or_none(baseline_seconds )
    candidate_seconds = finite_or_none(candidate_seconds)

    if baseline_seconds  is None or \
       candidate_seconds is None or \
       candidate_seconds <= 0:
        return None

    return baseline_seconds / candidate_seconds


def build_pmedian_model(
    distances             : np.ndarray,
    p                     : int,
    objective_upper_bound : float | int | None = None,
) -> tuple[Model, list[list], list]:
    if distances.ndim != 2:
        raise ValueError('Distances must be a 2D array.')

    n_rows, n_cols = distances.shape
    if n_rows != n_cols:
        raise ValueError('This implementation assumes a square distance matrix.')

    n = n_rows
    if not (1 <= p <= n):
        raise ValueError('P must satisfy 1 <= p <= n.')

    if np.any(distances < 0):
        raise ValueError('Distances must be nonnegative.')

    model = Model(solver_name='CBC')

    model.verbose = 0

    y = [
        model.add_var(
            var_type=BINARY, name=f'y_{j}'
        )
        for j in range(n)
    ]

    x: list[list]  = []
    row_cost_terms = []

    for i in range(n):
        x_row = [
            model.add_var(
                var_type=BINARY, name=f'x_{i}_{j}'
            )
            for j in range(n)
        ]
        x.append(x_row)

        model.add_constr(
            xsum(x_row[j] for j in range(n)) == 1, name=f'assign_{i}'
        )

        for j in range(n):
            model.add_constr(x_row[j] <= y[j], name=f'link_{i}_{j}')

        row_cost_terms.append(
            xsum(float(distances[i, j]) * x_row[j] for j in range(n))
        )

    model.add_constr(xsum(y[j] for j in range(n)) == p, name='select_p')

    objective_expr = xsum(row_cost_terms)

    objective_upper_bound = finite_or_none(
        objective_upper_bound
    )
    if objective_upper_bound is not None:
        model.add_constr(
            objective_expr <= objective_upper_bound,
            name='heuristic_upper_bound'           ,
        )

    model.objective = minimize(objective_expr)

    return model, x, y


def extract_open_facilities(y: list, threshold: float = 0.5) -> list[int]:
    return [
        index
        for index, variable in enumerate(y)
        if  variable.x is not None and variable.x >= threshold
    ]


def compute_solution_cost(
    distances       : np.ndarray,
    open_facilities : list[int] ,
) -> int | None:
    if not open_facilities:
        return None

    return int(distances[:, open_facilities].min(axis=1).sum())


def compute_assignments(
    distances       : np.ndarray,
    open_facilities : list[int] ,
) -> list[int]:
    if not open_facilities:
        return []

    closest_open_facility = np.argmin(
        distances[:, open_facilities], axis=1
    )

    return [
        open_facilities[index]
        for index in closest_open_facility.tolist()
    ]


def run_kmedoids(distances: np.ndarray, p: int) -> dict[str, object]:
    solve_start = perf_counter()

    estimator = KMedoids(
        n_clusters   =p,
        metric       ='precomputed'        ,
        method       =KMEDOIDS_METHOD      ,
        init         =KMEDOIDS_INIT        ,
        max_iter     =KMEDOIDS_MAX_ITER    ,
        random_state =KMEDOIDS_RANDOM_STATE,
    )

    estimator.fit(distances)

    solve_seconds = perf_counter() - solve_start

    open_facilities_zero_based = sorted(
        int(index)
        for index in estimator.medoid_indices_.tolist()
    )

    objective_value = compute_solution_cost(
        distances, open_facilities_zero_based
    )

    inertia_value = finite_or_none(getattr(estimator, 'inertia_', None))
    if objective_value is not None and inertia_value is not None:
        if abs(objective_value - inertia_value) > 1e-6:
            raise ValueError(
                'KMedoids inertia and validated objective do not match: '
                f'{inertia_value} vs {objective_value}.'
            )

    return {
        'objective_value'            : objective_value,
        'solve_seconds'              : solve_seconds  ,
        'n_iter'                     : getattr(estimator, 'n_iter_', None),
        'open_facilities_zero_based' : open_facilities_zero_based         ,
    }


def apply_mip_start(
    model           : Model     ,
    x               : list[list],
    y               : list      ,
    open_facilities : list[int] ,
    distances       : np.ndarray,
) -> None:
    assignments = compute_assignments(
        distances, open_facilities
    )

    start    = []
    open_set = set(open_facilities)

    for j, variable in enumerate(y):
        start.append((variable, 1.0 if j in open_set else 0.0))

    for i, x_row in enumerate(x):
        chosen_facility = assignments[i]

        for j, variable in enumerate(x_row):
            start.append((variable, 1.0 if j == chosen_facility else 0.0))

    model.start = start


def solve_instance_with_kmedoids_warm_start(
    instance_name      : str                     ,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    instance_path = INSTANCES_DIR / instance_name
    graph         = read_orlibrary_graph(instance_path)

    preprocess_start   = perf_counter()
    distances          = all_pairs_shortest_paths(graph['adjacency'])
    preprocess_seconds = perf_counter() - preprocess_start

    best_known_value         = finite_or_none(BEST_KNOWN_COSTS.get(instance_name))
    kmedoids_result          = run_kmedoids  (distances, graph['p'])
    kmedoids_objective_value = finite_or_none(kmedoids_result['objective_value'])

    if kmedoids_objective_value is not None and best_known_value is not None:
        if kmedoids_objective_value + 1e-6 < best_known_value:
            raise ValueError(
                'KMedoids objective is below the published best-known value: '
                f'{kmedoids_objective_value} < {best_known_value}.'
            )

    build_start = perf_counter()
    model, x, y = build_pmedian_model(
        distances ,
        graph['p'],
        objective_upper_bound=(
            kmedoids_objective_value if APPLY_HEURISTIC_CUTOFF else None
        ),
    )
    build_seconds = perf_counter() - build_start

    apply_mip_start(
        model,
        x    ,
        y    ,
        kmedoids_result['open_facilities_zero_based'], distances
    )

    solve_start   = perf_counter  ()
    status        = model.optimize(max_seconds=time_limit_seconds)
    solve_seconds = perf_counter  () - solve_start

    has_incumbent = status in {
        OptimizationStatus.OPTIMAL ,
        OptimizationStatus.FEASIBLE,
    }

    solver_objective_value = finite_or_none(model.objective_value if has_incumbent else None)
    objective_bound        = finite_or_none(model.objective_bound)

    open_facilities_zero_based: list[int] = []

    validated_objective_value = None

    if has_incumbent:
        open_facilities_zero_based = extract_open_facilities(y)

        if len(open_facilities_zero_based) != graph['p']:
            raise ValueError(
                f'Expected {graph["p"]} open facilities, but recovered {len(open_facilities_zero_based)}.'
            )

        validated_objective_value = compute_solution_cost(
            distances, open_facilities_zero_based
        )

        if solver_objective_value is not None and validated_objective_value is not None:
            if abs(solver_objective_value - validated_objective_value) > 1e-4:
                raise ValueError(
                    'Solver objective and validated objective do not match: '
                    f'{solver_objective_value} vs {validated_objective_value}.'
                )

        if validated_objective_value is not None and best_known_value is not None:
            if validated_objective_value + 1e-6 < best_known_value:
                raise ValueError(
                    'Validated objective is below the published best-known value: '
                    f'{validated_objective_value} < {best_known_value}.'
                )

    objective_value = validated_objective_value if validated_objective_value is not None else solver_objective_value

    kmedoids_open_facilities = [
        facility + 1
        for facility in kmedoids_result['open_facilities_zero_based']
    ]
    open_facilities = [
        facility + 1
        for facility in open_facilities_zero_based
    ]

    result = {
        'instance' : instance_name,
        'n'        : graph['n'],
        'm'        : graph['m'],
        'p'        : graph['p'],

        'best_known_value' : normalize_number(best_known_value),

        'kmedoids_method'                     : KMEDOIDS_METHOD      ,
        'kmedoids_init'                       : KMEDOIDS_INIT        ,
        'kmedoids_random_state'               : KMEDOIDS_RANDOM_STATE,
        'kmedoids_open_facilities_count'      : len(kmedoids_open_facilities               ),
        'kmedoids_open_facilities'            : ' '.join(map(str, kmedoids_open_facilities)),
        'kmedoids_solve_seconds'              : normalize_number(kmedoids_result['solve_seconds']),
        'kmedoids_n_iter'                     : normalize_number(kmedoids_result['n_iter']       ),
        'kmedoids_objective_value'            : normalize_number(kmedoids_objective_value        ),
        'kmedoids_gap_to_best_known_percent'  : normalize_number(gap_to_reference_percent(kmedoids_objective_value, best_known_value)),

        'warm_start_applied'      : True,
        'heuristic_cutoff_applied': APPLY_HEURISTIC_CUTOFF and kmedoids_objective_value is not None,

        'status'                   : getattr(status, 'name', str(status)),
        'objective_value'          : normalize_number(objective_value   ),
        'objective_bound'          : normalize_number(objective_bound   ),
        'solver_gap_percent'       : normalize_number(gap_to_bound_percent    (objective_value, objective_bound )),
        'gap_to_best_known_percent': normalize_number(gap_to_reference_percent(objective_value, best_known_value)),
        'matches_best_known'       : (
            objective_value  is not None and
            best_known_value is not None and
            abs(objective_value - best_known_value) < 1e-6
        ),

        'preprocess_seconds'  : normalize_number(preprocess_seconds           ),
        'model_build_seconds' : normalize_number(build_seconds                ),
        'solve_seconds'       : normalize_number(solve_seconds                ),
        'mip_stage_seconds'   : normalize_number(build_seconds + solve_seconds),
        'total_seconds'       : normalize_number(
            preprocess_seconds               +
            kmedoids_result['solve_seconds'] +
            build_seconds                    +
            solve_seconds
        ),

        'open_facilities_count' : len(open_facilities               ),
        'open_facilities'       : ' '.join(map(str, open_facilities)),

        'error': '',
    }

    del model
    del x
    del y
    del distances

    gc.collect()

    return result


def attempt_solve_instance_with_kmedoids_warm_start(
    instance_name      : str                     ,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    try:
        return solve_instance_with_kmedoids_warm_start(
            instance_name, time_limit_seconds=time_limit_seconds
        )
    except Exception as exc:
        gc.collect()

        return {
            'instance' : instance_name,
            'n'        : None,
            'm'        : None,
            'p'        : None,

            'best_known_value' : normalize_number(BEST_KNOWN_COSTS.get(instance_name)),

            'kmedoids_method'                    : KMEDOIDS_METHOD      ,
            'kmedoids_init'                      : KMEDOIDS_INIT        ,
            'kmedoids_random_state'              : KMEDOIDS_RANDOM_STATE,
            'kmedoids_n_iter'                    : None,
            'kmedoids_objective_value'           : None,
            'kmedoids_gap_to_best_known_percent' : None,
            'kmedoids_solve_seconds'             : None,
            'kmedoids_open_facilities_count'     : 0   ,
            'kmedoids_open_facilities'           : ''  ,

            'warm_start_applied'       : False  ,
            'heuristic_cutoff_applied' : False  ,
            'status'                   : 'ERROR',
            'objective_value'          : None   ,
            'objective_bound'          : None   ,
            'solver_gap_percent'       : None   ,
            'gap_to_best_known_percent': None   ,
            'matches_best_known'       : False  ,

            'preprocess_seconds'  : None,
            'model_build_seconds' : None,
            'solve_seconds'       : None,
            'mip_stage_seconds'   : None,
            'total_seconds'       : None,

            'open_facilities_count' : 0 ,
            'open_facilities'       : '',

            'error': str(exc),
        }

### APPLY

In [8]:
RESULTS = []

for index, instance_name in enumerate(INSTANCE_NAMES, start=1):
    print(f'[{index:02d}/{len(INSTANCE_NAMES):02d}] Solving {instance_name}...')

    result = attempt_solve_instance_with_kmedoids_warm_start(
        instance_name, time_limit_seconds=TIME_LIMIT_SECONDS
    )

    RESULTS.append(result)

    print(
        f"  kmedoids_obj={result['kmedoids_objective_value']},"
        f" kmedoids_gap={result['kmedoids_gap_to_best_known_percent']},"
        f" mip_status={result['status']},"
        f" mip_obj={result['objective_value']},"
        f" mip_gap={result['gap_to_best_known_percent']},"
        f" total_seconds={result['total_seconds']}"
    )

    if result['error']:
        print(f"  error={result['error']}")

RESULTS_DF = pd.DataFrame(RESULTS)

[01/40] Solving pmed1.txt...
  kmedoids_obj=5819, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=5819, mip_gap=0, total_seconds=1.2168
[02/40] Solving pmed2.txt...
  kmedoids_obj=4093, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=4093, mip_gap=0, total_seconds=2.0479
[03/40] Solving pmed3.txt...
  kmedoids_obj=4250, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=4250, mip_gap=0, total_seconds=1.9591
[04/40] Solving pmed4.txt...
  kmedoids_obj=3073, kmedoids_gap=1.2854, mip_status=OPTIMAL, mip_obj=3034, mip_gap=0, total_seconds=1.4814
[05/40] Solving pmed5.txt...
  kmedoids_obj=1401, kmedoids_gap=3.3948, mip_status=OPTIMAL, mip_obj=1355, mip_gap=0, total_seconds=1.4888
[06/40] Solving pmed6.txt...
  kmedoids_obj=7824, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=7824, mip_gap=0, total_seconds=21.2708
[07/40] Solving pmed7.txt...
  kmedoids_obj=5631, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=5631, mip_gap=0, total_seconds=2.8423
[08/40] Solving pmed8.txt...
  kmedoids_obj=4627, kmedoids_gap=4.

Saving the results:

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KMEDOIDS_RESULTS_DF = RESULTS_DF[
    [
        'instance',
        'n'       ,
        'm'       ,
        'p'       ,

        'best_known_value'     ,
        'kmedoids_method'      ,
        'kmedoids_init'        ,
        'kmedoids_random_state',
        'kmedoids_n_iter'      ,

        'kmedoids_objective_value'          ,
        'kmedoids_gap_to_best_known_percent',
        'kmedoids_solve_seconds'            ,
        'kmedoids_open_facilities_count'    ,
        'kmedoids_open_facilities'          ,

        'error',
    ]
].copy()

KMEDOIDS_RESULTS_DF.to_csv(KMEDOIDS_OUTPUT_PATH  , index=False)
RESULTS_DF         .to_csv(WARM_START_OUTPUT_PATH, index=False)

print(f'KMedoids results saved to      : {KMEDOIDS_OUTPUT_PATH  }')
print(f'Warm-start MIP results saved to: {WARM_START_OUTPUT_PATH}')

KMedoids results saved to      : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/kmedoids_results.csv
Warm-start MIP results saved to: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/mip_warm_start_results.csv


Displaying the results:

In [10]:
def print_table_summary(
    dataframe      : pd.DataFrame,
    label          : str,
    objective_col  : str,
    gap_col        : str,
    time_col       : str,
) -> None:
    solved_count  = int( dataframe[objective_col].notna().sum())
    matches_count = int((dataframe[gap_col      ].fillna(np.inf).abs() < 1e-6).sum())

    mean_time     = dataframe[time_col].dropna().mean()
    mean_gap      = dataframe[gap_col ].dropna().mean()

    print(label)
    print(f'  solved instances            : {solved_count }/{len(dataframe)}')
    print(f'  matches best-known          : {matches_count}/{len(dataframe)}')
    print(f'  mean time (s)               : {normalize_number(mean_time)}')
    print(f'  mean gap to best-known (%)  : {normalize_number(mean_gap )}')
    print()


print_table_summary(
    KMEDOIDS_RESULTS_DF,
    label        ='KMedoids summary:'       ,
    objective_col='kmedoids_objective_value',
    gap_col ='kmedoids_gap_to_best_known_percent',
    time_col='kmedoids_solve_seconds'            ,
)

print_table_summary(
    RESULTS_DF,
    label        ='Warm-start exact MIP summary:',
    objective_col='objective_value'              ,
    gap_col ='gap_to_best_known_percent',
    time_col='total_seconds'            ,
)
WARM_START_DISPLAY_DF = RESULTS_DF[
    [
        'instance',
        'best_known_value',
        'kmedoids_objective_value',
        'kmedoids_gap_to_best_known_percent',
        'kmedoids_solve_seconds',
        'status',
        'objective_value',
        'objective_bound',
        'solver_gap_percent',
        'gap_to_best_known_percent',
        'solve_seconds',
        'total_seconds',
    ]
].copy()

KMedoids summary:
  solved instances            : 40/40
  matches best-known          : 12/40
  mean time (s)               : 0.2646
  mean gap to best-known (%)  : 4.5236

Warm-start exact MIP summary:
  solved instances            : 35/40
  matches best-known          : 33/40
  mean time (s)               : 121.944
  mean gap to best-known (%)  : 0.004



Approach overview:

In [11]:
APPROACH_SUMMARY_DF = pd.DataFrame(
    [
        {
            'approach'                       : 'sklearn-extra KMedoids',
            'instances'                      : len(KMEDOIDS_RESULTS_DF),
            'mean_runtime_seconds'           : normalize_number(KMEDOIDS_RESULTS_DF['kmedoids_solve_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(KMEDOIDS_RESULTS_DF['kmedoids_gap_to_best_known_percent'].dropna().mean()),
        },
        {
            'approach'                       : 'Warm-start MIP (solve only)',
            'instances'                      : len(RESULTS_DF              ),
            'mean_runtime_seconds'           : normalize_number(RESULTS_DF['solve_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(RESULTS_DF['gap_to_best_known_percent'].dropna().mean()),
        },
        {
            'approach'                       : 'Warm-start MIP (full workflow)',
            'instances'                      : len(RESULTS_DF                 ),
            'mean_runtime_seconds'           : normalize_number(RESULTS_DF['total_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(RESULTS_DF['gap_to_best_known_percent'].dropna().mean()),
        },
    ]
)

display(APPROACH_SUMMARY_DF)

,approach,instances,mean_runtime_seconds,mean_gap_to_best_known_percent
0,sklearn-extra KMedoids,40,0.2646,4.5236
1,Warm-start MIP (solve only),40,117.8586,0.0040
2,Warm-start MIP (full workflow),40,121.9440,0.0040


KMedoids per-instance results:

In [12]:
display(KMEDOIDS_RESULTS_DF)

,instance,n,m,p,best_known_value,kmedoids_method,kmedoids_init,kmedoids_random_state,kmedoids_n_iter,kmedoids_objective_value,kmedoids_gap_to_best_known_percent,kmedoids_solve_seconds,kmedoids_open_facilities_count,kmedoids_open_facilities,error
0,pmed1.txt,100,200,5,5819,pam,k-medoids++,0,5,5819,0.0000,0.0066,5,7 13 65 91 99,
1,pmed2.txt,100,200,10,4093,pam,k-medoids++,0,8,4093,0.0000,0.0066,10,6 8 12 37 41 45 67 91 95 99,
2,pmed3.txt,100,200,10,4250,pam,k-medoids++,0,9,4250,0.0000,0.0073,10,5 9 13 21 26 36 48 55 69 99,
3,pmed4.txt,100,200,20,3034,pam,k-medoids++,0,9,3073,1.2854,0.0070,20,5 7 13 22 26 34 35 38 48 55 61 66 71 77 83 85 ...,
4,pmed5.txt,100,200,33,1355,pam,k-medoids++,0,9,1401,3.3948,0.0086,33,4 8 9 12 13 19 25 26 29 30 33 37 38 41 44 46 4...,
5,pmed6.txt,200,800,5,7824,pam,k-medoids++,0,5,7824,0.0000,0.0087,5,16 86 101 111 126,
6,pmed7.txt,200,800,10,5631,pam,k-medoids++,0,9,5631,0.0000,0.0181,10,3 10 72 87 131 142 181 186 191 199,
7,pmed8.txt,200,800,20,4445,pam,k-medoids++,0,9,4627,4.0945,0.0262,20,42 51 66 71 83 84 94 96 100 118 121 127 130 13...,
8,pmed9.txt,200,800,40,2734,pam,k-medoids++,0,9,2861,4.6452,0.0349,40,1 3 11 14 21 25 29 31 39 42 48 51 52 54 55 58 ...,
9,pmed10.txt,200,800,67,1255,pam,k-medoids++,0,9,1289,2.7092,0.0383,67,3 5 15 17 19 30 31 32 35 39 41 42 43 44 50 55 ...,


Warm-start exact MIP per-instance results:

In [13]:
display(WARM_START_DISPLAY_DF)

,instance,best_known_value,kmedoids_objective_value,kmedoids_gap_to_best_known_percent,kmedoids_solve_seconds,status,objective_value,objective_bound,solver_gap_percent,gap_to_best_known_percent,solve_seconds,total_seconds
0,pmed1.txt,5819,5819,0.0000,0.0066,OPTIMAL,5819.0,5.819000e+03,0.0000,0.0000,0.9608,1.2168
1,pmed2.txt,4093,4093,0.0000,0.0066,OPTIMAL,4093.0,4.093000e+03,0.0000,0.0000,1.8943,2.0479
2,pmed3.txt,4250,4250,0.0000,0.0073,OPTIMAL,4250.0,4.250000e+03,0.0000,0.0000,1.8495,1.9591
3,pmed4.txt,3034,3073,1.2854,0.0070,OPTIMAL,3034.0,3.034000e+03,0.0000,0.0000,1.3793,1.4814
4,pmed5.txt,1355,1401,3.3948,0.0086,OPTIMAL,1355.0,1.355000e+03,0.0000,0.0000,1.3821,1.4888
5,pmed6.txt,7824,7824,0.0000,0.0087,OPTIMAL,7824.0,7.824000e+03,0.0000,0.0000,20.8529,21.2708
6,pmed7.txt,5631,5631,0.0000,0.0181,OPTIMAL,5631.0,5.631000e+03,0.0000,0.0000,2.4361,2.8423
7,pmed8.txt,4445,4627,4.0945,0.0262,OPTIMAL,4445.0,4.445000e+03,0.0000,0.0000,13.3188,13.7306
8,pmed9.txt,2734,2861,4.6452,0.0349,OPTIMAL,2734.0,2.734000e+03,0.0000,0.0000,13.2193,13.6426
9,pmed10.txt,1255,1289,2.7092,0.0383,OPTIMAL,1255.0,1.255000e+03,0.0000,0.0000,13.4690,13.8969
